# CIFAR-10 Image Classification Model

This notebook loads and explores CIFAR-10, trains a compact convolutional neural network, evaluates it, and saves the Keras model and class labels for the FastAPI inference service.

In [2]:
import json
from pathlib import Path

import numpy as np
import tensorflow as tf

SEED = 42
tf.keras.utils.set_random_seed(SEED)

project_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    path for path in project_candidates if (path / "models").is_dir()
)
MODELS_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODELS_DIR / "my_classifier_model.h5"
LABELS_PATH = MODELS_DIR / "labels.json"

CLASS_LABELS = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

print(f"TensorFlow version: {tf.__version__}")
print(f"Model output path: {MODEL_PATH}")

TensorFlow version: 2.21.0
Model output path: c:\Users\shali\ml-prediction-api\models\my_classifier_model.h5


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

print(f"Training images: {x_train.shape}")
print(f"Training labels: {y_train.shape}")
print(f"Test images: {x_test.shape}")
print(f"Test labels: {y_test.shape}")

class_counts = np.bincount(y_train, minlength=len(CLASS_LABELS))
for label, count in zip(CLASS_LABELS, class_counts):
    print(f"{label}: {count}")

sample_indices = [np.where(y_train == class_index)[0][0] for class_index in range(len(CLASS_LABELS))]
print(f"Sample image shape: {x_train[sample_indices[0]].shape}")
print(f"Sample labels: {[CLASS_LABELS[y_train[index]] for index in sample_indices]}")

   204800/170498071 ━━━━━━━━━━━━━━━━━━━━ 1:16:07 27us/step

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print(f"Normalized training range: {x_train.min():.1f} to {x_train.max():.1f}")
print(f"Normalized test range: {x_test.min():.1f} to {x_test.max():.1f}")

In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(len(CLASS_LABELS), activation="softmax"),
    ],
    name="cifar10_classifier",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    restore_best_weights=True,
)

history = model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=8,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1,
)

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")

In [ ]:
model.save(MODEL_PATH)
LABELS_PATH.write_text(json.dumps(CLASS_LABELS, indent=2), encoding="utf-8")

print(f"Saved model: {MODEL_PATH}")
print(f"Saved labels: {LABELS_PATH}")

In [ ]:
loaded_model = tf.keras.models.load_model(MODEL_PATH)
loaded_labels = json.loads(LABELS_PATH.read_text(encoding="utf-8"))

assert MODEL_PATH.exists() and MODEL_PATH.stat().st_size > 0
assert LABELS_PATH.exists() and loaded_labels == CLASS_LABELS
assert loaded_model.output_shape[-1] == len(loaded_labels)

sample_predictions = loaded_model.predict(x_test[:3], verbose=0)
print(f"Model artifact size: {MODEL_PATH.stat().st_size:,} bytes")
print(f"Labels: {loaded_labels}")
print(f"Prediction tensor shape: {sample_predictions.shape}")
print("Model artifacts verified successfully.")